In [ ]:
# =========================
# Latent Memory Compression
# Stable Dependency Setup
# =========================

!pip install -q \
torch==2.3.1 \
transformers==4.41.2 \
datasets==2.19.2 \
accelerate==0.31.0 \
sentencepiece==0.2.0 \
huggingface_hub==0.23.5 \
fsspec==2024.3.1 \
tqdm==4.66.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 885.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.8/402.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import transformers
import datasets
import huggingface_hub
import fsspec

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)

torch: 2.3.1+cu121
transformers: 4.41.2
datasets: 2.19.2
hub: 0.23.5
fsspec: 2024.3.1


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
from torch.utils.data import (
    Dataset,
    DataLoader,
)
from transformers.modeling_outputs import BaseModelOutput

from tqdm.auto import tqdm

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

for p in model.parameters():
    p.requires_grad = False

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded.


In [ ]:
import torch
import torch.nn as nn


class PerceiverBlock(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_heads,
        ff_mult=4,
    ):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True,
        )

        self.self_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True,
        )

        self.ff = nn.Sequential(
            nn.Linear(
                hidden_size,
                hidden_size * ff_mult,
            ),
            nn.GELU(),
            nn.Linear(
                hidden_size * ff_mult,
                hidden_size,
            ),
        )

        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.norm3 = nn.LayerNorm(hidden_size)

    def forward(
        self,
        latents,
        hidden_states,
    ):
        cross_out, _ = self.cross_attn(
            query=latents,
            key=hidden_states,
            value=hidden_states,
        )

        latents = self.norm1(
            latents + cross_out
        )

        self_out, _ = self.self_attn(
            query=latents,
            key=latents,
            value=latents,
        )

        latents = self.norm2(
            latents + self_out
        )

        ff_out = self.ff(latents)

        latents = self.norm3(
            latents + ff_out
        )

        return latents


class PerceiverCompressor(nn.Module):
    def __init__(
        self,
        hidden_size=768,
        num_latents=128,
        num_heads=12,
        depth=4,
    ):
        super().__init__()

        self.latents = nn.Parameter(
            torch.randn(
                num_latents,
                hidden_size,
            )
        )

        self.blocks = nn.ModuleList(
            [
                PerceiverBlock(
                    hidden_size,
                    num_heads,
                )
                for _ in range(depth)
            ]
        )

    def forward(
        self,
        hidden_states,
    ):
        batch_size = hidden_states.size(0)

        latents = self.latents.unsqueeze(0).expand(
            batch_size,
            -1,
            -1,
        )

        for block in self.blocks:
            latents = block(
                latents,
                hidden_states,
            )

        return latents

In [ ]:
compressor = PerceiverCompressor(hidden_size=768,
    num_latents=128,
    num_heads=12,
    depth=2,).to(device)

print("Initialized new Perceiver compressor.")

Checkpoint loaded.


In [ ]:
TEMPERATURE = 2.0
MSE_WEIGHT = 1000

def distillation_kl(
    student_logits,
    teacher_logits,
    temperature=2.0,
):

    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=-1,
    )

    teacher_probs = F.softmax(
        teacher_logits / temperature,
        dim=-1,
    )

    loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    )

    loss *= temperature ** 2

    return loss

In [ ]:
def training_step(
    input_ids,
    attention_mask,
):

    decoder_input_ids = model._shift_right(
        input_ids
    )

    with torch.no_grad():

        teacher_encoder = model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        teacher_hidden = (
            teacher_encoder.last_hidden_state
        )

        teacher_outputs = model(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=teacher_hidden
            ),
            decoder_input_ids=decoder_input_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        teacher_logits = teacher_outputs.logits

        teacher_decoder_hidden = (
            teacher_outputs.decoder_hidden_states[-1]
        )

    compressed = compressor(
        teacher_hidden
    )

    student_outputs = model(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=compressed
        ),
        decoder_input_ids=decoder_input_ids,
        output_hidden_states=True,
        return_dict=True,
    )

    student_logits = student_outputs.logits

    student_decoder_hidden = (
        student_outputs.decoder_hidden_states[-1]
    )

    kl_loss = distillation_kl(
        student_logits,
        teacher_logits,
        TEMPERATURE,
    )

    mse_loss = F.mse_loss(
        student_decoder_hidden,
        teacher_decoder_hidden,
    )

    total_loss = (
        kl_loss
        + MSE_WEIGHT * mse_loss
    )

    return (
        total_loss,
        kl_loss.detach(),
        mse_loss.detach(),
    )

In [ ]:
compressor.train()

optimizer = torch.optim.AdamW(
    compressor.parameters(),
    lr=1e-5,
    weight_decay=1e-4,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train"
)

train_texts = []

for x in dataset:

    txt = x["text"].strip()

    if len(
        tokenizer(
            txt,
            truncation=False
        )["input_ids"]
    ) >= 256:

        train_texts.append(txt)

    if len(train_texts) >= 50000:
        break

print("training samples:", len(train_texts))


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/6407814 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10315 > 512). Running this sequence through the model will result in indexing errors


training samples: 50000


In [ ]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):

    def __init__(
        self,
        texts,
        tokenizer,
        max_length=256,
    ):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
             max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].squeeze(0)

        attention_mask = enc[
            "attention_mask"
        ].squeeze(0)

        labels = input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

In [ ]:
train_dataset = TextDataset(
    train_texts,
    tokenizer,
    max_length=256,
)

print(len(train_dataset))

50000


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
)

In [ ]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

loss, kl, mse = training_step(
    input_ids,
    attention_mask,
)

print("Loss:", loss.item())
print("KL:", kl.item())
print("MSE:", mse.item())

Loss: 5278.61767578125
KL: 5215.04296875
MSE: 0.06357458233833313


In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):

    compressor.train()

    running_loss = 0

    pbar = tqdm(train_loader)

    for batch in pbar:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        loss, kl, mse = training_step(
            input_ids,
            attention_mask,

        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            compressor.parameters(),
            1.0,
        )

        optimizer.step()

        running_loss += loss.item()

        pbar.set_description(
            f"loss={loss.item():.4f} "
            f"kl={kl.item():.4f} "
            f"mse={mse.item():.4f}"
        )

    print(
        f"Epoch {epoch+1}: "
        f"{running_loss/len(train_loader):.4f}"
    )

  0%|          | 0/6250 [00:00<?, ?it/s]

Epoch 1: 3258.7282


  0%|          | 0/6250 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
torch.save(
    compressor.state_dict(),
    "compressor_hybrid_kl_mse.pt",
)

print("Saved.")

Saved.


In [ ]:
def compare_generation(text):

    compressor.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(device)

    with torch.no_grad():

        encoder_hidden = model.encoder(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
        ).last_hidden_state

        compressed = compressor(
            encoder_hidden
        )

        teacher = model.generate(
            **inputs,
            max_new_tokens=40,
        )

        student = model.generate(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=compressed
            ),
            attention_mask=torch.ones(
                compressed.shape[:2],
                dtype=torch.long,
                device=device,
            ),
            max_new_tokens=40,
        )

    print(
        "Teacher:",
        tokenizer.decode(
            teacher[0],
            skip_special_tokens=True,
        ),
    )

    print()

    print(
        "Student:",
        tokenizer.decode(
            student[0],
            skip_special_tokens=True,
        ),
    )

In [ ]:
import torch
import torch.nn.functional as F

from transformers.modeling_outputs import BaseModelOutput

def evaluate_batch(
    input_ids,
    attention_mask,
):

    decoder_input_ids = model._shift_right(
        input_ids
    )

    with torch.no_grad():

        # Teacher

        teacher_encoder = model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        teacher_hidden = (
            teacher_encoder.last_hidden_state
        )

        teacher_outputs = model(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=teacher_hidden
            ),
            decoder_input_ids=decoder_input_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        teacher_logits = teacher_outputs.logits

        teacher_decoder_hidden = (
            teacher_outputs.decoder_hidden_states[-1]
        )

        # Student

        compressed = compressor(
            teacher_hidden
        )

        student_outputs = model(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=compressed
            ),
            decoder_input_ids=decoder_input_ids,
            output_hidden_states=True,
            return_dict=True,
        )

        student_logits = student_outputs.logits

        student_decoder_hidden = (
            student_outputs.decoder_hidden_states[-1]
        )

    return (
        teacher_logits,
        student_logits,
        teacher_decoder_hidden,
        student_decoder_hidden,
    )

In [ ]:
def compute_metrics(
    teacher_logits,
    student_logits,
    teacher_hidden,
    student_hidden,
):

    metrics = {}

    # Hidden MSE

    metrics["hidden_mse"] = F.mse_loss(
        student_hidden,
        teacher_hidden,
    ).item()

    # Cosine

    metrics["cosine"] = F.cosine_similarity(
        teacher_hidden.flatten(),
        student_hidden.flatten(),
        dim=0,
    ).item()

    # Norms

    metrics["teacher_norm"] = (
        teacher_hidden.norm(dim=-1)
        .mean()
        .item()
    )

    metrics["student_norm"] = (
        student_hidden.norm(dim=-1)
        .mean()
        .item()
    )

    # Top-1

    teacher_top1 = teacher_logits.argmax(
        dim=-1
    )

    student_top1 = student_logits.argmax(
        dim=-1
    )

    metrics["top1"] = (
        (
            teacher_top1
            ==
            student_top1
        )
        .float()
        .mean()
        .item()
    )

    # Top-5

    teacher_top5 = torch.topk(
        teacher_logits,
        k=5,
        dim=-1,
    ).indices

    student_top5 = torch.topk(
        student_logits,
        k=5,
        dim=-1,
    ).indices

    top5_match = (
        teacher_top5
        ==
        student_top5
    ).any(-1)

    metrics["top5"] = (
        top5_match.float()
        .mean()
        .item()
    )

    # Top-5 overlap

    overlap = []

    for i in range(
        teacher_top5.size(0)
    ):
        for j in range(
            teacher_top5.size(1)
        ):

            a = set(
                teacher_top5[i,j]
                .cpu()
                .tolist()
            )

            b = set(
                student_top5[i,j]
                .cpu()
                .tolist()
            )

            overlap.append(
                len(a & b) / 5
            )

    metrics["top5_overlap"] = (
        sum(overlap)
        / len(overlap)
    )

    # KL

    teacher_probs = F.softmax(
        teacher_logits,
        dim=-1,
    )

    student_log_probs = (
        F.log_softmax(
            student_logits,
            dim=-1,
        )
    )

    metrics["kl"] = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean",
    ).item()

    # Entropy

    teacher_entropy = (
        -(teacher_probs *
          torch.log(
              teacher_probs
              + 1e-8
          ))
        .sum(-1)
        .mean()
    )

    student_probs = F.softmax(
        student_logits,
        dim=-1,
    )

    student_entropy = (
        -(student_probs *
          torch.log(
              student_probs
              + 1e-8
          ))
        .sum(-1)
        .mean()
    )

    metrics["teacher_entropy"] = (
        teacher_entropy.item()
    )

    metrics["student_entropy"] = (
        student_entropy.item()
    )

    # Logit stats

    metrics["teacher_logit_mean"] = (
        teacher_logits.mean().item()
    )

    metrics["teacher_logit_std"] = (
        teacher_logits.std().item()
    )

    metrics["student_logit_mean"] = (
        student_logits.mean().item()
    )

    metrics["student_logit_std"] = (
        student_logits.std().item()
    )

    return metrics

In [ ]:
compressor.eval()

all_metrics = []

for idx in range(100):

    batch = next(iter(train_loader))

    input_ids = batch[
        "input_ids"
    ].to(device)

    attention_mask = batch[
        "attention_mask"
    ].to(device)

    (
        teacher_logits,
        student_logits,
        teacher_hidden,
        student_hidden,
    ) = evaluate_batch(
        input_ids,
        attention_mask,
    )

    m = compute_metrics(
        teacher_logits,
        student_logits,
        teacher_hidden,
        student_hidden,
    )

    all_metrics.append(m)

In [ ]:
keys = all_metrics[0].keys()

for k in keys:

    avg = sum(
        x[k]
        for x in all_metrics
    ) / len(all_metrics)

    print(
        f"{k:25s} {avg:.4f}"
    )